# 05 · Generación de Respuesta
Construye una respuesta en lenguaje natural a partir de los fragmentos
recuperados, incluyendo **citas** con metadatos completos
(`doc_id`, `página`, `chunk_id`).

### ¿Por qué Gemini aquí?
La generación de respuestas largas y coherentes con citas académicas
requiere un modelo capaz de:
- Sintetizar múltiples fragmentos sin contradecirse.
- Seguir instrucciones complejas de formato (citas numeradas).
- Manejar contextos largos (hasta 8 000 tokens de contexto).

Gemini 2.0 Flash cumple todos estos criterios superando a Groq en
tareas de síntesis informativa extendida.


In [ ]:
!pip install -q \
    langchain==0.3.14 \
    langchain-community==0.3.14 \
    langchain-google-genai==2.0.8 \
    faiss-cpu==1.9.0
print('✓ Dependencias instaladas')


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

BASE_DIR  = '/content/drive/MyDrive/RAG_BioMed'
INDEX_DIR = f'{BASE_DIR}/faiss_index'

GEMINI_KEY   = userdata.get('GEMINI_API_KEY')
GEMINI_MODEL = 'gemini-2.0-flash'
EMBED_MODEL  = 'models/text-embedding-004'


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

def cargar_vs(index_dir, api_key):
    emb = GoogleGenerativeAIEmbeddings(model=EMBED_MODEL, google_api_key=api_key)
    return FAISS.load_local(index_dir, emb, allow_dangerous_deserialization=True)

vs = cargar_vs(INDEX_DIR, GEMINI_KEY)
print('✓ Vectorstore cargado')


In [ ]:
def formatear_contexto(docs: list[Document]) -> tuple[str, list[dict]]:
    """
    Construye el bloque de contexto para el prompt y la lista de citas.

    Cada fragmento se etiqueta con un número [N] que el modelo puede
    referenciar en su respuesta. Los metadatos conservados son:
        doc_id, titulo, autores, página, chunk_id, similarity_score.

    Args:
        docs : Documentos recuperados por la búsqueda semántica.

    Returns:
        (contexto_str, lista_de_citas)
    """
    partes, citas = [], []
    for i, doc in enumerate(docs, 1):
        m = doc.metadata
        encabezado = (
            f'[{i}] {m.get("titulo","")[:60]}  |  '
            f'doc_id={m.get("doc_id")}  '
            f'p.{m.get("page",0)+1}  '
            f'chunk={m.get("chunk_id","")}'
        )
        partes.append(f'--- Fragmento {i} ---\n{encabezado}\n{doc.page_content}')
        citas.append({
            'numero'          : i,
            'doc_id'          : m.get('doc_id'),
            'titulo'          : m.get('titulo', '')[:80],
            'autores'         : m.get('autores', ''),
            'pagina'          : m.get('page', 0) + 1,
            'chunk_id'        : m.get('chunk_id', ''),
            'similarity_score': m.get('similarity_score', 0.0),
        })
    return '\n\n'.join(partes), citas


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

def crear_generador(api_key: str):
    """
    Construye la cadena de generación de respuestas basadas en contexto.

    Usa Gemini 2.0 Flash con temperatura baja para respuestas precisas
    pero con algo de fluidez en la redacción.

    Args:
        api_key : Google Gemini API key.

    Returns:
        Función `generar(query, docs) -> dict`.
    """
    llm = ChatGoogleGenerativeAI(
        model=GEMINI_MODEL,
        google_api_key=api_key,
        temperature=0.2
    )
    prompt = ChatPromptTemplate.from_messages([
        ('system', """Eres un asistente experto en bioinformática y medicina basado
en un corpus de artículos científicos de arXiv.

INSTRUCCIONES:
1. Responde ÚNICAMENTE con información del contexto proporcionado.
2. Incluye citas numéricas [1], [2], etc. cuando uses cada fragmento.
3. Si la información no está en el contexto, indícalo con: "No encontrado en el corpus".
4. Mantén un tono académico y preciso.
5. Estructura la respuesta con párrafos claros.

CONTEXTO:
{contexto}"""),
        ('human', '{query}')
    ])
    chain = prompt | llm

    def generar(query: str, docs: list[Document],
                sugerencias: str = '') -> dict:
        """
        Genera respuesta con citas a partir del contexto recuperado.

        Args:
            query       : Pregunta del usuario.
            docs        : Documentos recuperados.
            sugerencias : Feedback del verificador para regenerar (opcional).

        Returns:
            Dict con respuesta, citas y contexto usado.
        """
        contexto, citas = formatear_contexto(docs)
        prompt_extra = f'\n\nMEJORA SOLICITADA: {sugerencias}' if sugerencias else ''
        resp = chain.invoke({'query': query + prompt_extra, 'contexto': contexto[:6000]})
        return {
            'query'              : query,
            'respuesta'          : resp.content,
            'citas'              : citas,
            'num_fragmentos'     : len(docs),
            'contexto_completo'  : contexto,
        }

    return generar


In [ ]:
# ── Demo ──────────────────────────────────────────────────────────────────
generar = crear_generador(GEMINI_KEY)

query = '¿Cómo se aplica el deep learning para el diagnóstico de enfermedades en imágenes médicas?'
docs  = vs.similarity_search(query, k=5)
resultado = generar(query, docs)

print('=== RESPUESTA GENERADA ===')
print(resultado['respuesta'])

print('\n=== CITAS ===')
for c in resultado['citas']:
    print(f'  [{c["numero"]}] {c["doc_id"]} | p.{c["pagina"]} | '
          f'{c["titulo"][:50]}  (score={c["similarity_score"]})')
